# Light Pre-process before labeling

In [ ]:
!pip install deep-translator

# Upload CSV
from google.colab import files
uploaded = files.upload()

import pandas as pd
from deep_translator import GoogleTranslator

# Load CSV mentah
filename = next(iter(uploaded))
df = pd.read_csv(filename, on_bad_lines='skip', encoding='utf-8', quoting=3)

# Ambil kolom yang diperlukan + ubah ke tanggal
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce').dt.date
df = df[['created_at', 'full_text']]

# Hapus duplikat berdasarkan full_text
df = df.drop_duplicates(subset='full_text')

# Translate ke English
def translate_to_english(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(str(text))
    except:
        return text

df['full_text_en'] = df['full_text'].apply(translate_to_english)

# Simpan hanya yang dibutuhkan untuk labeling manual
out_path = '/content/labeling_ready_translated.csv'
df[['created_at', 'full_text_en']].to_csv(out_path, index=False)

files.download(out_path)


Saving 7-10-2023 - 7-04-2025 All In.csv to 7-10-2023 - 7-04-2025 All In (1).csv


/tmp/ipython-input-8-742616852.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce').dt.date


✅ Siap untuk labeling. Duplikat dihapus, teks sudah diterjemahkan.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Pre-process after labeling

### Pre-Process Stopword, Stemming, Lemitization

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from google.colab import files

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# 1. Upload file manual
print("⬆Upload file CSV `:")
uploaded = files.upload()

# Ambil nama file
filename = next(iter(uploaded))

# 2. Load dan baca data
try:
    df = pd.read_csv(filename, delimiter=';', quoting=3, encoding='utf-8', on_bad_lines='skip')
    print("✅ Dibaca dengan UTF-8")
except UnicodeDecodeError:
    try:
        df = pd.read_csv(filename, delimiter=';', quoting=3, encoding='ISO-8859-1', on_bad_lines='skip')
        print("✅ Dibaca dengan ISO-8859-1")
    except UnicodeDecodeError:
        df = pd.read_csv(filename, delimiter=';', quoting=3, encoding='latin1', on_bad_lines='skip')
        print("✅ Dibaca dengan latin1 (fallback)")

# 3. Inisialisasi tools preprocessing
stop_words = set(stopwords.words('english'))
sentiment_stopwords = {
    "the", "a", "an", "and", "or", "but", "if", "while", "of", "at", "by", "for", "with", "about",
    "against", "between", "into", "through", "during", "before", "after", "above", "below", "to",
    "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then",
    "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few",
    "more", "most", "other", "some", "such", "only", "own", "same", "so", "than", "too", "very",
    "can", "will", "just", "should", "now", "you", "i", "me", "my", "we", "us", "our", "they",
    "them", "their", "he", "him", "his", "she", "her", "it", "its", "this", "that", "these", "those"
}
custom_stopwords = stop_words - sentiment_stopwords

stemmer = PorterStemmer()
sensitive_words = {"israel", "palestine", "hamas", "genocide", "gaza", "zionist"}

def custom_stem_id(word):
    word_lower = word.lower()
    if word_lower in sensitive_words:
        return word_lower
    else:
        return stemmer.stem(word_lower)

# lemmatizer = WordNetLemmatizer()

# 4. Fungsi preprocessing lengkap
def full_preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)      # hapus URL
    text = re.sub(r"@\w+", "", text)                         # hapus mention
    text = re.sub(r"[^\w\s]", "", text)                      # hapus tanda baca
    text = re.sub(r"[\U00010000-\U0010ffff]", "", text)      # hapus emoji
    text = re.sub(r"\d+", "", text)                          # hapus angka
    text = re.sub(r"\s+", " ", text).strip()                 # hapus spasi berlebih
    text = re.sub(r'[^\x00-\x7F]+', '', str(text))           # hapus simbol non-ASCII

    # Tokenisasi dan stopword removal
    words = word_tokenize(text)

    words = [word for word in words if word not in stop_words]

    # Lematisasi dan stemming
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    stemmed_tokens = [custom_stem_id(w) for w in words]

    # Return teks yang sudah dibersihkan
    return " ".join(stemmed_tokens)

# 5. Jalankan preprocessing
df = df[df['full_text_processed'].notna()]
df['full_text_processed'] = df['full_text_processed'].apply(full_preprocess)

# 6. Simpan hasil
output_file = 'final_preprocessed.csv'
if 'created_at' in df.columns:
    df[['created_at', 'full_text_processed', 'label']].to_csv(output_file, index=False)
else:
    df[['full_text_processed', 'label']].to_csv(output_file, index=False)

print(f"Preprocessing selesai. File disimpan sebagai `{output_file}`.")
files.download(output_file)

df.head()


⬆Upload file CSV `:


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Saving fixed_final_dataset - Copy.csv to fixed_final_dataset - Copy (1).csv
✅ Dibaca dengan ISO-8859-1
✅ Preprocessing selesai. File disimpan sebagai `final_preprocessed.csv`.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,created_at,full_text_processed,label,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,06/04/2025,anybodi need chip insert brain name fafo satan...,0.0,NaN,NaN,NaN,NaN,NaN
1,06/04/2025,pledg support justic digniti peac gaza join st...,0.0,NaN,NaN,NaN,NaN,NaN
2,06/04/2025,take posit gaza take posit gaza one live rest ...,0.0,NaN,NaN,NaN,NaN,NaN
3,06/04/2025,children gaza stand proof world look away dont...,0.0,NaN,NaN,NaN,NaN,NaN
4,06/04/2025,donat famili gaza struggl surviv war everi lit...,0.0,NaN,NaN,NaN,NaN,NaN
